In [1]:
# examples/demo_semua_momen.py

import mortapy as mp
import math
import os # Pastikan os diimpor

# Fungsi helper
def tampilkan_hasil_momen(deskripsi_awal: str, hasil_objek: mp.ActuarialResult, verifikasi_manual_str: str = ""):
    print(deskripsi_awal)
    hasil_objek.show()
    if verifikasi_manual_str:
        print(verifikasi_manual_str)
    print("-" * 60)

# --- Definisi Parameter Umum ---
print("=" * 70)
print("        DEMO LENGKAP FUNGSI MOMEN FUTURE LIFETIME MORTAPY")
print("=" * 70)

usia_x = 65
suku_bunga_untuk_kalkulator = 0.05 # Tidak selalu dipakai untuk momen murni, tapi API butuh
gender_pilihan_tabel = 'wanita'
n_temporary = 10 # Untuk momen temporary

# Parameter untuk Asumsi
qx_konstan_val = 0.03
omega_dm_val = 100.0
alpha_beta_val = 1.5 # Untuk General De Moivre (Beta Distribution)
mu_cfm_val = 0.025
gompertz_params_val = [0.0001, 1.1] # B, c
makeham_params_val = [0.0002, 0.00008, 1.12] # A, B, c

print("\n--- Parameter Umum yang Digunakan dalam Demo ---")
print(f"Usia awal (x)         : {usia_x}")
print(f"Suku bunga (i)        : {suku_bunga_untuk_kalkulator:.2%}")
print(f"Gender (untuk tabel)  : {gender_pilihan_tabel.capitalize()}")
print(f"Periode temporary (n) : {n_temporary} tahun")
print("-" * 60)

# ==============================================================================
# BAGIAN 1: MOMEN BERBASIS TABEL MORTALITA (TMI DEFAULT)
# ==============================================================================
print("\n" + "=" * 70)
print(" BAGIAN 1: MOMEN BERBASIS TABEL MORTALITA (TMI DEFAULT)")
print("=" * 70)

try:
    tabel_default = mp.load_default_table()
    print(f"\nBerhasil memuat tabel default: {tabel_default}\n")
    print(f"--- Menggunakan parameter: Usia = {usia_x}, Gender = {gender_pilihan_tabel}, n_temporary = {n_temporary} ---")

    # --- 1.1 Momen Curtate ---
    print("\n--- 1.1 Momen Curtate Future Lifetime (Tabel) ---")
    ex_wl_tabel = mp.ex_curtate_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel)
    tampilkan_hasil_momen(f"a. Ekspektasi Curtate (e_{{{usia_x}}}, Whole Life):", ex_wl_tabel)

    ex_temp_tabel = mp.ex_curtate_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary)
    tampilkan_hasil_momen(f"b. Ekspektasi Curtate (e_{{{usia_x}:\\overline{{{n_temporary}}}|}}, Temporary):", ex_temp_tabel)

    e_sq_wl_tabel = mp.e_sq_curtate_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel)
    tampilkan_hasil_momen(f"c. Momen Kedua Curtate (E[K_{{{usia_x}}}^2], Whole Life):", e_sq_wl_tabel)

    e_sq_temp_tabel = mp.e_sq_curtate_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary)
    tampilkan_hasil_momen(f"d. Momen Kedua Curtate (E[K_{{{usia_x}:\\overline{{{n_temporary}}}|}}^2], Temporary):", e_sq_temp_tabel)

    var_k_wl_tabel = mp.var_k_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel)
    tampilkan_hasil_momen(f"e. Variansi Curtate (Var[K_{{{usia_x}}}], Whole Life):", var_k_wl_tabel,
                         verifikasi_manual_str=f"    -> Verifikasi: {e_sq_wl_tabel.value:.4f} - ({ex_wl_tabel.value:.4f})^2 = {e_sq_wl_tabel.value - ex_wl_tabel.value**2:.4f}")

    var_k_temp_tabel = mp.var_k_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary)
    tampilkan_hasil_momen(f"f. Variansi Curtate (Var[K_{{{usia_x}:\\overline{{{n_temporary}}}|}}], Temporary):", var_k_temp_tabel,
                          verifikasi_manual_str=f"    -> Verifikasi: {e_sq_temp_tabel.value:.4f} - ({ex_temp_tabel.value:.4f})^2 = {e_sq_temp_tabel.value - ex_temp_tabel.value**2:.4f}")

    # --- 1.2 Momen Complete (dengan asumsi UDD untuk bagian fraksional) ---
    print("\n--- 1.2 Momen Complete Future Lifetime (Tabel, Aproksimasi UDD Fraksional) ---")
    ex_c_wl_tabel = mp.ex_complete_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, assumption_fractional='udd')
    tampilkan_hasil_momen(f"a. Ekspektasi Complete (e_circ_{{{usia_x}}}, Whole Life):", ex_c_wl_tabel,
                         verifikasi_manual_str=f"    -> Aproksimasi UDD: {ex_wl_tabel.value:.4f} + 0.5 = {ex_wl_tabel.value + 0.5:.4f}")

    ex_c_temp_tabel = mp.ex_complete_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary, assumption_fractional='udd')
    tampilkan_hasil_momen(f"b. Ekspektasi Complete (e_circ_{{{usia_x}:\\overline{{{n_temporary}}}|}}, Temporary):", ex_c_temp_tabel)

    e_sq_c_wl_tabel = mp.e_sq_complete_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, assumption_fractional='udd')
    tampilkan_hasil_momen(f"c. Momen Kedua Complete (E[T_{{{usia_x}}}^2], Whole Life):", e_sq_c_wl_tabel,
                           verifikasi_manual_str=f"    -> Aproksimasi UDD: {e_sq_wl_tabel.value:.4f} + {ex_wl_tabel.value:.4f} + 1/3 = {e_sq_wl_tabel.value + ex_wl_tabel.value + (1/3):.4f}")
    
    e_sq_c_temp_tabel = mp.e_sq_complete_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary, assumption_fractional='udd')
    tampilkan_hasil_momen(f"d. Momen Kedua Complete (E[T_{{{usia_x}:\\overline{{{n_temporary}}}|}}^2], Temporary):", e_sq_c_temp_tabel)

    var_t_wl_tabel = mp.var_t_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, assumption_fractional='udd')
    tampilkan_hasil_momen(f"e. Variansi Complete (Var[T_{{{usia_x}}}], Whole Life):", var_t_wl_tabel,
                         verifikasi_manual_str=f"    -> Verifikasi: {e_sq_c_wl_tabel.value:.4f} - ({ex_c_wl_tabel.value:.4f})^2 = {e_sq_c_wl_tabel.value - ex_c_wl_tabel.value**2:.4f}")

    var_t_temp_tabel = mp.var_t_table(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary, assumption_fractional='udd')
    tampilkan_hasil_momen(f"f. Variansi Complete (Var[T_{{{usia_x}:\\overline{{{n_temporary}}}|}}], Temporary):", var_t_temp_tabel,
                          verifikasi_manual_str=f"    -> Verifikasi: {e_sq_c_temp_tabel.value:.4f} - ({ex_c_temp_tabel.value:.4f})^2 = {e_sq_c_temp_tabel.value - ex_c_temp_tabel.value**2:.4f}")


except FileNotFoundError as e:
    print(f"\n[PERINGATAN] Gagal memuat tabel mortalita default: {e}")
    print("Bagian 1 demo (momen berbasis tabel) akan dilewati.")
except Exception as e_table:
    print(f"\n[ERROR] Terjadi kesalahan pada perhitungan momen berbasis tabel: {e_table}")


# ==============================================================================
# BAGIAN 2: MOMEN BERBASIS ASUMSI DISTRIBUSI MURNI
# ==============================================================================
print("\n" + "=" * 70)
print(" BAGIAN 2: MOMEN BERBASIS ASUMSI DISTRIBUSI MURNI")
print("=" * 70)

assumptions_to_test_moments = [
    ('constant_qx', [qx_konstan_val], f"q_x konstan = {qx_konstan_val}"),
    ('de_moivre', [omega_dm_val], f"De Moivre (ω={int(omega_dm_val)})"),
    ('beta_distribution', [omega_dm_val, alpha_beta_val], f"Beta Dist. (ω={int(omega_dm_val)}, α={alpha_beta_val})"),
    ('gompertz', gompertz_params_val, f"Gompertz (B={gompertz_params_val[0]:.2e}, c={gompertz_params_val[1]})"),
    ('makeham', makeham_params_val, f"Makeham (A={makeham_params_val[0]:.2e}, B={makeham_params_val[1]:.2e}, c={makeham_params_val[2]})")
]

for i, (assumption_type, params, desc_short) in enumerate(assumptions_to_test_moments):
    print(f"\n--- 2.{i+1} Menggunakan Asumsi: {desc_short} ---")
    print(f"   Parameter Umum: Usia = {usia_x}, n_temporary = {n_temporary}, Suku Bunga = {suku_bunga_untuk_kalkulator:.2%}")
    print(f"   Parameter Asumsi: {params}\n")

    try:
        # --- Momen Curtate ---
        print(f"   --- Momen Curtate (K_x) ---")
        ex_wl_as = mp.ex_curtate_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil_momen(f"   a. Ekspektasi (e_{{{usia_x}}}, WL):", ex_wl_as)
        ex_temp_as = mp.ex_curtate_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params, n_temp=n_temporary) # type: ignore
        tampilkan_hasil_momen(f"   b. Ekspektasi (e_{{{usia_x}:\\overline{{{n_temporary}}}|}}, Temp):", ex_temp_as)
        
        e_sq_wl_as = mp.e_sq_curtate_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil_momen(f"   c. Momen Kedua (E[K_{{{usia_x}}}^2], WL):", e_sq_wl_as)
        e_sq_temp_as = mp.e_sq_curtate_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params, n_temp=n_temporary) # type: ignore
        tampilkan_hasil_momen(f"   d. Momen Kedua (E[K_{{{usia_x}:\\overline{{{n_temporary}}}|}}^2], Temp):", e_sq_temp_as)

        var_k_wl_as = mp.var_k_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil_momen(f"   e. Variansi (Var[K_{{{usia_x}}}], WL):", var_k_wl_as, verifikasi_manual_str=f"      -> Verifikasi: {e_sq_wl_as.value:.4f} - ({ex_wl_as.value:.4f})^2 = {e_sq_wl_as.value - ex_wl_as.value**2:.4f}")
        var_k_temp_as = mp.var_k_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params, n_temp=n_temporary) # type: ignore
        tampilkan_hasil_momen(f"   f. Variansi (Var[K_{{{usia_x}:\\overline{{{n_temporary}}}|}}], Temp):", var_k_temp_as, verifikasi_manual_str=f"      -> Verifikasi: {e_sq_temp_as.value:.4f} - ({ex_temp_as.value:.4f})^2 = {e_sq_temp_as.value - ex_temp_as.value**2:.4f}")

        # --- Momen Complete ---
        print(f"\n   --- Momen Complete (T_x) ---")
        ex_c_wl_as = mp.ex_complete_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil_momen(f"   g. Ekspektasi (e_circ_{{{usia_x}}}, WL):", ex_c_wl_as)
        ex_c_temp_as = mp.ex_complete_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params, n_temp=n_temporary) # type: ignore
        tampilkan_hasil_momen(f"   h. Ekspektasi (e_circ_{{{usia_x}:\\overline{{{n_temporary}}}|}}, Temp):", ex_c_temp_as)

        e_sq_c_wl_as = mp.e_sq_complete_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil_momen(f"   i. Momen Kedua (E[T_{{{usia_x}}}^2], WL):", e_sq_c_wl_as)
        e_sq_c_temp_as = mp.e_sq_complete_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params, n_temp=n_temporary) # type: ignore
        tampilkan_hasil_momen(f"   j. Momen Kedua (E[T_{{{usia_x}:\\overline{{{n_temporary}}}|}}^2], Temp):", e_sq_c_temp_as)

        var_t_wl_as = mp.var_t_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params) # type: ignore
        tampilkan_hasil_momen(f"   k. Variansi (Var[T_{{{usia_x}}}], WL):", var_t_wl_as, verifikasi_manual_str=f"      -> Verifikasi: {e_sq_c_wl_as.value:.4f} - ({ex_c_wl_as.value:.4f})^2 = {e_sq_c_wl_as.value - ex_c_wl_as.value**2:.4f}")
        var_t_temp_as = mp.var_t_assumption(age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, assumption_type=assumption_type, params=params, n_temp=n_temporary) # type: ignore
        tampilkan_hasil_momen(f"   l. Variansi (Var[T_{{{usia_x}:\\overline{{{n_temporary}}}|}}], Temp):", var_t_temp_as, verifikasi_manual_str=f"      -> Verifikasi: {e_sq_c_temp_as.value:.4f} - ({ex_c_temp_as.value:.4f})^2 = {e_sq_c_temp_as.value - ex_c_temp_as.value**2:.4f}")

    except Exception as e_assume:
        print(f"   [ERROR] Terjadi kesalahan pada asumsi {assumption_type} untuk momen: {e_assume}")


print("\n" + "=" * 70)
print("              DEMO LENGKAP MORTAPY SELESAI")
print("=" * 70)

        DEMO LENGKAP FUNGSI MOMEN FUTURE LIFETIME MORTAPY

--- Parameter Umum yang Digunakan dalam Demo ---
Usia awal (x)         : 65
Suku bunga (i)        : 5.00%
Gender (untuk tabel)  : Wanita
Periode temporary (n) : 10 tahun
------------------------------------------------------------

 BAGIAN 1: MOMEN BERBASIS TABEL MORTALITA (TMI DEFAULT)

Berhasil memuat tabel default: <MortalityTable ultimate='tabel_mortalita_penduduk_indonesia_2023.csv'>

--- Menggunakan parameter: Usia = 65, Gender = wanita, n_temporary = 10 ---

--- 1.1 Momen Curtate Future Lifetime (Tabel) ---
a. Ekspektasi Curtate (e_{65}, Whole Life):


<IPython.core.display.Math object>

Deskripsi: Ekspektasi Curtate Future Lifetime (Tabel), Usia 65, Gender Wanita
------------------------------------------------------------
b. Ekspektasi Curtate (e_{65:\overline{10}|}, Temporary):


<IPython.core.display.Math object>

Deskripsi: Ekspektasi Curtate Future Lifetime 10-thn temp (Tabel), Usia 65, Gender Wanita
------------------------------------------------------------
c. Momen Kedua Curtate (E[K_{65}^2], Whole Life):


...

Deskripsi: Momen Kedua Complete Future Lifetime 10-tahun temporary (Asumsi: Makeham (A=0.0002, B=8e-05, c=1.12) (Frac: UDD)), Usia 65
------------------------------------------------------------
   k. Variansi (Var[T_{65}], WL):


<IPython.core.display.Math object>

Deskripsi: Variansi Complete Future Lifetime (Makeham (A=0.0002, B=8e-05, c=1.12), Usia 65
      -> Verifikasi: 36.1757 - (4.8864)^2 = 12.2992
------------------------------------------------------------
   l. Variansi (Var[T_{65:\overline{10}|}], Temp):


<IPython.core.display.Math object>

Deskripsi: Variansi Complete Future Lifetime 10-tahun temporary (Makeham (A=0.0002, B=8e-05, c=1.12), Usia 65
      -> Verifikasi: 32.4656 - (4.6883)^2 = 10.4856
------------------------------------------------------------

              DEMO LENGKAP MORTAPY SELESAI
